# LULUCF and vegetation flux summaries by country (adm0)

Reads the global per-country summary parquet (`summary_global/adm0/global_summary_by_adm0.parquet`) and writes two CSVs, one for `flux_source == "LULUCF"` and one for `flux_source == "vegetation"`.

Each CSV sums `value` and `area_ha` over these columns, dropping the land state detail columns (`land_state_broad_class`, `tall_veg_type`, `land_state_detailed_class`, `land_state_meaning`):

`adm0`, `country_name`, `flux_source`, `IPCC_summary`, `IPCC_node_code`, `year`, `IPCC_change`, `IPCC_class`, `IPCC_class_name`, `starting_composite_primary_forest`, `analysis_layer`

`country_name` is added from `cn.iso_to_country`. Set `repo_root` below so `src.utilities` can be imported. Reading the `s3://` parquet uses `s3fs`.

In [1]:
from pathlib import Path
import gc
import sys

import pandas as pd

# Root of the AFOLU flux model repo (the folder that contains `src/`), so the shared utilities import.
repo_root = Path(r"/mnt/c/GitHub/AFOLU_flux_model")  # Update if needed
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

In [2]:
# Update these if needed.

# Folder name of the LULUCF run. Spelled out because cn points at newer model versions than this run.
lulucf_run_folder = "LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1"

adm0_dir = (f"{cn.LULUCF_dir}outputs_LULUCF_totals/{lulucf_run_folder}/zonal_stats/land_use/summary_global/adm0")

input_path = f"{adm0_dir}/global_summary_by_adm0.parquet"
output_dir = adm0_dir

# CSVs are saved here first, then uploaded to output_dir
local_output_dir = Path(r"/mnt/c/GIS/afolu_model/land_use/zonal_statistics/summary_adm0")
local_output_dir.mkdir(parents=True, exist_ok=True)

flux_sources = ["LULUCF", "vegetation"]

print("Input:", input_path)
print("Local output folder:", local_output_dir)
print("S3 output folder:", output_dir)

Input: s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_LULUCF_totals/LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1/zonal_stats/land_use/summary_global/adm0/global_summary_by_adm0.parquet
Local output folder: /mnt/c/GIS/afolu_model/land_use/zonal_statistics/summary_adm0
S3 output folder: s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_LULUCF_totals/LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1/zonal_stats/land_use/summary_global/adm0


In [3]:
GROUP_COLS = [
    cn.adm0_pattern,
    "country_name",
    "flux_source",
    "IPCC_summary",
    "IPCC_node_code",
    "year",
    "IPCC_change",
    "IPCC_class",
    "IPCC_class_name",
    "starting_composite_primary_forest",
    "analysis_layer",
]
VALUE_COLS = ["value", "area_ha"]
OUTPUT_COLS = GROUP_COLS + VALUE_COLS

# Columns read from the parquet (country_name is added here)
READ_COLS = [c for c in GROUP_COLS if c != "country_name"] + VALUE_COLS

# Small territories that aren't in cn.iso_to_country
EXTRA_COUNTRY_NAMES = {
    "ASM": "American Samoa",
    "ATA": "Antarctica",
    "COK": "Cook Islands",
    "GUM": "Guam",
    "MHL": "Marshall Islands",
    "PYF": "French Polynesia",
    "TON": "Tonga",
    "WLF": "Wallis and Futuna",
    "WSM": "Samoa",
}

# Same short names the zonal stats step uses
COUNTRY_NAME_RENAMES = {
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Russian Federation": "Russia",
    "Democratic Republic of the Congo": "DR Congo",
    "United States of America (the)": "USA",
}


def log(message):
    print(f"{uu.timestr()}: {message}", flush=True)


def add_country_name(df):
    adm0 = df[cn.adm0_pattern].astype(str)

    # "NA" is the zonal stats placeholder for pixels outside any country (not Namibia, which is NAM)
    adm0 = adm0.replace({"NA": "Unassigned", "": "Unassigned", "nan": "Unassigned"})
    df[cn.adm0_pattern] = adm0

    names = {**cn.iso_to_country, **EXTRA_COUNTRY_NAMES}
    unique_codes = pd.Series(adm0.unique())
    code_to_name = dict(zip(unique_codes, unique_codes.map(names)))
    code_to_name["Unassigned"] = "Unassigned"

    missing = sorted(c for c, n in code_to_name.items() if pd.isna(n))
    if missing:
        log(f"WARNING: no country name for {missing}; using the adm0 code as the name")
        for c in missing:
            code_to_name[c] = c

    df["country_name"] = adm0.map(code_to_name).replace(COUNTRY_NAME_RENAMES)
    return df


def summarize_flux_source(input_path, flux_source):
    log(f"Reading {flux_source} rows")
    df = pd.read_parquet(
        input_path,
        columns=READ_COLS,
        filters=[("flux_source", "==", flux_source)],
    )
    log(f"{flux_source}: {len(df):,} input rows")

    if df.empty:
        raise ValueError(f"No rows with flux_source == {flux_source!r} in {input_path}")

    df = add_country_name(df)

    # Values are float32 in the parquet; sum in float64 so totals don't lose precision
    df[VALUE_COLS] = df[VALUE_COLS].astype("float64")
    input_totals = df.groupby("analysis_layer")[VALUE_COLS].sum()

    summary = (
        df.groupby(GROUP_COLS, dropna=False, observed=True, as_index=False)[VALUE_COLS]
          .sum()
    )
    del df
    gc.collect()

    summary = summary[OUTPUT_COLS].sort_values(GROUP_COLS).reset_index(drop=True)

    # Dropping the land state columns only merges rows, so totals per layer must be unchanged
    output_totals = summary.groupby("analysis_layer")[VALUE_COLS].sum()
    diff = (output_totals - input_totals).abs().max().max()
    log(f"{flux_source}: {len(summary):,} output rows; max total difference vs input = {diff:,.6f}")

    return summary, input_totals


def write_csv(summary, local_output_dir, output_dir, flux_source, timestamp):
    file_name = f"global_summary_by_adm0__{flux_source}__{timestamp}.csv"
    local_path = Path(local_output_dir) / file_name
    s3_path = f"{output_dir}/{file_name}"

    summary.to_csv(local_path, index=False)
    log(f"Saved {local_path}")

    uu.upload_s3_file(s3_path, str(local_path))
    log(f"Uploaded {s3_path}")
    return s3_path

## Summarize and write one CSV per flux source

In [4]:
timestamp = uu.timestr()

summaries = {}
csv_paths = {}

for flux_source in flux_sources:
    summary, totals = summarize_flux_source(input_path, flux_source)
    summaries[flux_source] = summary
    csv_paths[flux_source] = write_csv(summary, local_output_dir, output_dir, flux_source, timestamp)

csv_paths

20261014_22_55_25: Reading LULUCF rows
20261014_22_55_47: LULUCF: 10,863,579 input rows
20261014_22_55_54: LULUCF: 766,850 output rows; max total difference vs input = 0.000000
20261014_22_56_04: Saved /mnt/c/GIS/afolu_model/land_use/zonal_statistics/summary_adm0/global_summary_by_adm0__LULUCF__20261014_22_55_25.csv
20261014_22_56_30: Uploaded s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_LULUCF_totals/LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1/zonal_stats/land_use/summary_global/adm0/global_summary_by_adm0__LULUCF__20261014_22_55_25.csv
20261014_22_56_30: Reading vegetation rows
20261014_22_56_46: vegetation: 2,649,583 input rows
20261014_22_56_47: vegetation: 204,980 output rows; max total difference vs input = 0.000000
20261014_22_56_50: Saved /mnt/c/GIS/afolu_model/land_use/zonal_statistics/summary_adm0/global_summary_by_adm0__vegetation__20261014_22_55_25.csv
20261014_22_56_57: Uploaded s3://gfw2-data/climate/AFOLU_flux_model/LULUC

{'LULUCF': 's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_LULUCF_totals/LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1/zonal_stats/land_use/summary_global/adm0/global_summary_by_adm0__LULUCF__20261014_22_55_25.csv',
 'vegetation': 's3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_LULUCF_totals/LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1/zonal_stats/land_use/summary_global/adm0/global_summary_by_adm0__vegetation__20261014_22_55_25.csv'}

If only the upload failed, the CSVs are already in `local_output_dir`. After fixing credentials, run this cell to upload them without recalculating.

In [5]:
for flux_source in flux_sources:
    file_name = f"global_summary_by_adm0__{flux_source}__{timestamp}.csv"
    local_path = local_output_dir / file_name
    if local_path.exists():
        uu.upload_s3_file(f"{output_dir}/{file_name}", str(local_path))
        log(f"Uploaded {output_dir}/{file_name}")
    else:
        log(f"Not found locally: {local_path}")


20261014_22_57_24: Uploaded s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_LULUCF_totals/LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1/zonal_stats/land_use/summary_global/adm0/global_summary_by_adm0__LULUCF__20261014_22_55_25.csv
20261014_22_57_31: Uploaded s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_LULUCF_totals/LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1/zonal_stats/land_use/summary_global/adm0/global_summary_by_adm0__vegetation__20261014_22_55_25.csv


## QA

In [6]:
for flux_source, summary in summaries.items():
    print(f"\n{flux_source}: {len(summary):,} rows, {summary[cn.adm0_pattern].nunique()} countries")
    display(summary.groupby("analysis_layer")[VALUE_COLS].sum())


LULUCF: 766,850 rows, 235 countries


,value,area_ha
analysis_layer,,
LULUCF_gross_emissions__all_C_pools__all_gases__MgCO2e,"183,609,446,451.935","141,216,925,728.821"
LULUCF_gross_removals__all_C_pools__MgCO2,"-215,606,260,310.476","140,561,804,860.960"
LULUCF_net_flux__all_C_pools__all_gases__MgCO2e,"-31,996,813,882.954","141,333,604,004.865"



vegetation: 204,980 rows, 235 countries


,value,area_ha
analysis_layer,,
net_flux__all_C_pools__all_gases__MgCO2e,"-66,677,742,883.561","43,585,535,807.557"


In [7]:
# Totals by country and year for the main net flux layer of each source
for flux_source, summary in summaries.items():
    net_layer = [l for l in summary["analysis_layer"].unique() if "net_flux" in l]
    print(f"\n{flux_source}: {net_layer}")
    display(
        summary[summary["analysis_layer"].isin(net_layer)]
        .groupby([cn.adm0_pattern, "country_name", "year"])[VALUE_COLS].sum()
        .head(20)
    )


LULUCF: ['LULUCF_net_flux__all_C_pools__all_gases__MgCO2e']


value         area_ha
adm0 country_name year                                
ABW  Aruba        2016      -1,310.044      18,194.522
                  2017      -1,282.158      18,195.274
                  2018      -1,310.026      18,194.447
                  2019      -1,361.617      18,193.244
                  2020      -1,375.965      18,194.447
                  2021      -1,379.449      18,194.522
                  2022      -1,282.041      18,194.823
                  2023      -1,287.901      18,193.921
                  2024      -1,031.270      18,192.569
AFG  Afghanistan  2016  16,630,905.852  64,383,824.833
                  2017  15,437,725.819  64,383,825.931
                  2018  15,267,841.468  64,383,821.401
                  2019  14,450,538.120  64,383,815.937
                  2020  15,070,920.243  64,383,820.076
                  2021  15,862,600.218  64,383,814.962
                  2022  15,891,026.210  64,383,822.462
                  2023  16,193,139.357  64,383,819.780
                  2024  14,289,707.268  64,383,823.584
AGO  Angola       2016  94,147,876.078 124,691,195.742
                  2017 112,883,580.600 124,691,183.541


vegetation: ['net_flux__all_C_pools__all_gases__MgCO2e']


value        area_ha
adm0 country_name year                               
ABW  Aruba        2016      -1,262.314        227.578
                  2017      -1,234.428        227.127
                  2018      -1,262.296        224.797
                  2019      -1,313.887        222.166
                  2020      -1,328.234        223.368
                  2021      -1,331.718        222.843
                  2022      -1,234.310        220.738
                  2023      -1,240.171        229.081
                  2024        -983.540        246.143
AFG  Afghanistan  2016  -1,827,375.735    663,860.334
                  2017  -3,020,556.069    825,734.269
                  2018  -3,190,439.785    839,137.107
                  2019  -4,007,743.860    882,783.696
                  2020  -3,387,361.992    897,878.862
                  2021  -2,595,716.910    794,574.500
                  2022  -2,567,291.453    780,866.003
                  2023  -2,265,177.889    764,776.510
                  2024  -4,168,609.941  1,133,267.621
AGO  Angola       2016  84,364,346.346 90,465,728.980
                  2017 103,100,041.443 89,862,002.388

In [8]:
summaries["LULUCF"].head(10)

,adm0,country_name,flux_source,IPCC_summary,IPCC_node_code,year,IPCC_change,IPCC_class,IPCC_class_name,starting_composite_primary_forest,analysis_layer,value,area_ha
0,ABW,Aruba,LULUCF,11,10,2016,11,1,Settlements,0,LULUCF_gross_emissions__all_C_pools__all_gases...,"3,261.992","9,730.054"
1,ABW,Aruba,LULUCF,11,10,2016,11,1,Settlements,0,LULUCF_gross_removals__all_C_pools__MgCO2,"-4,731.402","9,730.655"
2,ABW,Aruba,LULUCF,11,10,2016,11,1,Settlements,0,LULUCF_net_flux__all_C_pools__all_gases__MgCO2e,"-1,469.410","9,730.956"
3,ABW,Aruba,LULUCF,11,10,2016,11,1,Settlements,1,LULUCF_gross_emissions__all_C_pools__all_gases...,20.719,53.960
4,ABW,Aruba,LULUCF,11,10,2016,11,1,Settlements,1,LULUCF_gross_removals__all_C_pools__MgCO2,-32.419,54.036
5,ABW,Aruba,LULUCF,11,10,2016,11,1,Settlements,1,LULUCF_net_flux__all_C_pools__all_gases__MgCO2e,-11.700,54.036
6,ABW,Aruba,LULUCF,11,10,2017,11,1,Settlements,0,LULUCF_gross_emissions__all_C_pools__all_gases...,"3,292.273","9,730.054"
7,ABW,Aruba,LULUCF,11,10,2017,11,1,Settlements,0,LULUCF_gross_removals__all_C_pools__MgCO2,"-4,717.638","9,729.979"
8,ABW,Aruba,LULUCF,11,10,2017,11,1,Settlements,0,LULUCF_net_flux__all_C_pools__all_gases__MgCO2e,"-1,425.364","9,730.956"
9,ABW,Aruba,LULUCF,11,10,2017,11,1,Settlements,1,LULUCF_gross_emissions__all_C_pools__all_gases...,20.719,53.960
